### 분석: 전체 데이터 파일, 각 폴더 파일 수 확인

In [ ]:
import os
import json
import math
import random
import pandas as pd
from PIL import Image
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import defaultdict
from pathlib import Path

data_dir = "../data/raw"

folder_info = []

# data 바로 아래 폴더 3개만 탐색
for folder in os.listdir(data_dir):
    folder_path = os.path.join(data_dir, folder)

    if os.path.isdir(folder_path):
        file_count = sum(len(files) for _, _, files in os.walk(folder_path))

        folder_info.append({"폴더명": folder, "파일 수": file_count})

df = pd.DataFrame(folder_info)

print(f"전체 파일 수: {df['파일 수'].sum():,}개")
display(df)

### 분석: 전체 데이터 이미지 크기가 동일한지 확인

In [ ]:
for folder in ["train_images", "test_images"]:
    folder_path = os.path.join(data_dir, folder)
    image_sizes = []

    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith((".jpg", ".jpeg", ".png")):
                with Image.open(os.path.join(root, file)) as img:
                    image_sizes.append(img.size)

    size_counts = Counter(image_sizes)

    print(f"\n[{folder}]")
    print(f"전체 이미지 수: {len(image_sizes):,}개")
    print(f"모든 이미지 크기 동일 여부: {len(size_counts) == 1}")

    for size, count in size_counts.most_common():
        print(f"  {size}: {count:,}개")

### 분석: 이미지와 어노테이션 사이의 매칭 누락이 있는지 확인

In [ ]:
train_dir = "../data/raw/train_images"
annotation_dir = "../data/raw/train_annotations"

# 1. train_images 안의 실제 이미지 파일명 수집
image_files = set()

for root, dirs, files in os.walk(train_dir):
    for file in files:
        if file.lower().endswith((".png", ".jpg", ".jpeg")):
            image_files.add(file)

# 2. JSON 하나씩 확인하면서 매칭 여부 검사
total_json = 0
matched_json = 0
unmatched_json = []

for root, dirs, files in os.walk(annotation_dir):
    for file in files:
        if file.lower().endswith(".json"):
            total_json += 1
            json_path = os.path.join(root, file)

            with open(json_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            # images 리스트에서 file_name 추출
            image_list = data.get("images", [])

            if len(image_list) == 0:
                unmatched_json.append((file, "images 정보 없음"))
                continue

            file_name = image_list[0].get("file_name")

            if file_name in image_files:
                matched_json += 1
            else:
                unmatched_json.append((file, file_name))

# 3. 결과 출력
print(f"전체 JSON 수: {total_json}개")
print(f"정상적으로 이미지와 매칭된 JSON 수: {matched_json}개")
print(f"매칭 실패한 JSON 수: {len(unmatched_json)}개")

if len(unmatched_json) == 0:
    print("\n✅ 763개의 JSON이 모두 train_images 안의 232개 이미지와 정상 매칭됩니다.")
else:
    print("\n⚠️ 매칭 실패한 JSON 예시:")
    for item in unmatched_json[:10]:
        print(item)

### 분석: Bounding Box 시각화(랜덤 4장)로 확인하기

In [ ]:
# Windows 한글 폰트 설정
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 이미지 파일 경로 수집
image_paths = {}

for root, dirs, files in os.walk(train_dir):
    for file in files:
        if file.lower().endswith((".png", ".jpg", ".jpeg")):
            image_paths[file] = os.path.join(root, file)

# 이미지별 Bounding Box와 약품명 수집
image_annotations = defaultdict(list)

for root, dirs, files in os.walk(annotation_dir):
    for file in files:
        if file.lower().endswith(".json"):
            json_path = os.path.join(root, file)

            with open(json_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            for img_info in data.get("images", []):
                image_name = img_info["file_name"]
                drug_name = img_info.get("dl_name", "Unknown")

                for ann in data.get("annotations", []):
                    bbox = ann.get("bbox")

                    if bbox is not None and len(bbox) == 4:
                        image_annotations[image_name].append((bbox, drug_name))

# 랜덤 이미지 4장 선택
sample_images = random.sample(list(image_annotations.keys()), min(4, len(image_annotations)))

fig, axes = plt.subplots(2, 2, figsize=(12, 14))

for ax, image_name in zip(axes.flatten(), sample_images):
    image_path = image_paths[image_name]

    with Image.open(image_path) as img:
        ax.imshow(img.copy())

    for bbox, drug_name in image_annotations[image_name]:
        # COCO 형식: [x, y, width, height]
        x, y, w, h = bbox

        rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor="red", facecolor="none")
        ax.add_patch(rect)

        ax.text(
            x,
            y - 5,
            drug_name,
            fontsize=9,
            color="red",
            bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"),
        )

    ax.set_title(f"Bounding Boxes: {len(image_annotations[image_name])}", fontsize=12)
    ax.axis("off")

for ax in axes.flatten()[len(sample_images) :]:
    ax.axis("off")

plt.tight_layout()
plt.show()

### 분석: bbox 오류 확인 및 수정 완료

학습 이미지 232장과 JSON 763개에 들어 있는 bbox 763개를 검사했다.
검사 당시 동일 이미지 내 중복 bbox 3건과 이미지 범위 초과 1건을 확인했다.

| 문제 이미지 조합 | 수정한 알약 코드 | 수정 전 bbox | 수정 후 bbox |
|---|---|---|---|
| 001900-016548-019607-033009 (70) | K-016548 | [88, 255, 366, 209] | [102, 862, 234, 236] |
| 003351-029667-031863 (70) | K-003351 | [178, 249, 205, 203] | [372, 858, 188, 190] |
| 003351-020238-031863 (70) | K-020238 | [446, 844, 190, 191] | [584, 286, 220, 224] |
| 003351-016262-018357 (75) | K-018357 | [6567, 625, 311, 315] | [570, 622, 310, 318] |

서로 다른 알약 종류에 동일 좌표가 붙은 3건은 실제 대상 알약 위치로 박스를 다시 지정했다.
나머지 1건은 시작 x좌표 6567이 이미지 너비 976을 넘으므로 대상 캡슐 위치에 재지정했다.
수정 좌표는 이미지 경계를 육안으로 확인해 지정한 값이며, bbox와 area(너비 × 높이)만 변경했다.
오류가 발생한 작업 단계나 원인은 원본 배포본 및 작업 이력이 없어 확정하지 않았다.

- 수정 전 백업: `../reports/originals/`
- 수정본 4개 사본: `../reports/corrected_annotations/`
- 변경 기록: `../reports/bbox_repair/changes.json`
- 실제 사용 라벨: `../data/raw/train_annotations/` (수정 적용됨)

#### 수정 전: 백업 좌표를 사용한 2×2 시각화

아래 셀은 현재 라벨을 읽은 뒤 수정한 4건만 백업 JSON 좌표로 대체한다.
시각화를 위한 메모리상의 대체이며 파일을 변경하지 않는다.


In [ ]:
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

# 확인할 문제 이미지 4장
targets = [
    "K-001900-016548-019607-033009_0_2_0_2_70_000_200.png",
    "K-003351-029667-031863_0_2_0_2_70_000_200.png",
    "K-003351-020238-031863_0_2_0_2_70_000_200.png",
    "K-003351-016262-018357_0_2_0_2_75_000_200.png",
]

image_paths = {
    path.name: path for path in (root / "data/raw/train_images").rglob("*") if path.name in targets
}
annotations = defaultdict(list)

for path in (root / "data/raw/train_annotations").rglob("*.json"):
    with path.open(encoding="utf-8") as file:
        data = json.load(file)

    images = {info["id"]: info for info in data.get("images", [])}
    for ann in data.get("annotations", []):
        info = images.get(ann.get("image_id"))
        if info and info["file_name"] in targets:
            annotations[info["file_name"]].append((info["drug_N"], ann["bbox"]))

# 수정한 항목만 원본 백업 좌표로 되돌려 수정 전 상태 재현
backup_paths = list((root / "reports/originals").glob("*.json.bak"))
if len(backup_paths) != 4:
    raise FileNotFoundError("수정 전 JSON 백업 4개를 확인하세요.")

for backup_path in backup_paths:
    with backup_path.open(encoding="utf-8-sig") as file:
        backup = json.load(file)
    backup_images = {info["id"]: info for info in backup["images"]}
    backup_groups = defaultdict(list)
    for ann in backup["annotations"]:
        info = backup_images[ann["image_id"]]
        backup_groups[(info["file_name"], info["drug_N"])].append(ann["bbox"])
    for (image_name, drug_code), boxes in backup_groups.items():
        annotations[image_name] = [
            (code, bbox) for code, bbox in annotations[image_name] if code != drug_code
        ] + [(drug_code, bbox) for bbox in boxes]

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

for ax, name in zip(axes.flat, targets):
    with Image.open(image_paths[name]) as img:
        ax.imshow(img)
        image_width, image_height = img.size

    print(f"\n{name}\n이미지 크기: {image_width} × {image_height}")
    seen = {}
    max_right, max_bottom = image_width, image_height
    has_outside = False

    for index, (drug_code, bbox) in enumerate(annotations[name]):
        x, y, width, height = bbox
        key = tuple(bbox)
        duplicate = key in seen
        outside = x < 0 or y < 0 or x + width > image_width or y + height > image_height

        color = plt.get_cmap("tab10")(index % 10)
        ax.add_patch(
            patches.Rectangle(
                (x, y),
                width,
                height,
                edgecolor=color,
                facecolor="none",
                linewidth=3 if duplicate else 2,
                linestyle="--" if duplicate else "-",
            )
        )

        message = f"{drug_code}: {bbox}"
        if duplicate:
            message += f" ← {seen[key]}와 완전히 겹침"
        if outside:
            message += " ← 이미지 밖"
            has_outside = True

        print(message)
        ax.text(
            0.01,
            -0.06 - index * 0.055,
            message,
            transform=ax.transAxes,
            fontsize=9,
            color=color,
        )

        seen.setdefault(key, drug_code)
        max_right = max(max_right, x + width)
        max_bottom = max(max_bottom, y + height)

    # 이미지 밖 박스까지 포함해 실제 거리 표시
    ax.set_xlim(-20, max_right + 50)
    ax.set_ylim(max_bottom + 50, -20)
    ax.set_title(
        f"수정 전: {name}\n"
        + ("이미지 밖 박스까지 표시" if has_outside else "실선·점선이 같은 위치면 중복"),
        fontsize=9,
    )
    ax.set_xlabel("x 좌표")
    ax.set_ylabel("y 좌표")

plt.tight_layout(h_pad=7)
plt.show()

#### 수정 후: 현재 JSON을 다시 읽어 2×2 시각화

각 알약에 박스가 하나씩 붙고 이미지 안에 위치하는지 확인한다.


In [ ]:
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

targets = [
    "K-001900-016548-019607-033009_0_2_0_2_70_000_200.png",
    "K-003351-029667-031863_0_2_0_2_70_000_200.png",
    "K-003351-020238-031863_0_2_0_2_70_000_200.png",
    "K-003351-016262-018357_0_2_0_2_75_000_200.png",
]

# 수정된 JSON에서 각 이미지의 모든 박스 수집
annotations = defaultdict(list)

for path in (root / "data/raw/train_annotations").rglob("*.json"):
    with path.open(encoding="utf-8") as file:
        data = json.load(file)

    images = {info["id"]: info for info in data.get("images", [])}

    for ann in data.get("annotations", []):
        info = images.get(ann.get("image_id"))
        if info and info["file_name"] in targets:
            annotations[info["file_name"]].append(
                (info["drug_N"], ann["bbox"])
            )

image_paths = {
    path.name: path
    for path in (root / "data/raw/train_images").rglob("*")
    if path.name in targets
}

fig, axes = plt.subplots(2, 2, figsize=(12, 14))

for ax, image_name in zip(axes.flat, targets):
    with Image.open(image_paths[image_name]) as img:
        ax.imshow(img)
        image_width, image_height = img.size

    for index, (drug_code, bbox) in enumerate(annotations[image_name]):
        x, y, width, height = bbox
        color = plt.get_cmap("tab10")(index % 10)

        ax.add_patch(
            patches.Rectangle(
                (x, y), width, height,
                linewidth=2, edgecolor=color, facecolor="none",
            )
        )
        ax.text(
            x, y - 8, drug_code,
            fontsize=9, color=color,
            bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "none"},
        )

    ax.set_xlim(0, image_width)
    ax.set_ylim(image_height, 0)
    ax.set_title(f"수정 후 Bounding Boxes: {len(annotations[image_name])}")
    ax.axis("off")

plt.tight_layout()
plt.show()

#### 재검사 결과 및 해석

수정 후 학습 이미지 232장, JSON 763개, bbox 763개를 다시 검사했다.
중복 좌표와 이미지 범위 초과를 포함한 검사 코드의 오류 항목은 0건이었다.
수정한 4개 박스를 이미지 위에 표시해 대상 알약을 감싸는 것도 확인했다.

이번 수정은 라벨 좌표를 바로잡은 작업이다. 이미지 크기, 알약 종류 ID, 라벨 개수는 바꾸지 않았다.
정상 범위의 좌표가 다른 알약을 가리키거나 실제 알약의 라벨이 누락된 경우까지
전체 데이터에서 모두 해결되었다는 의미는 아니다.

검사 결과는 `../reports/bbox_audit/summary.json`, 수정 전 결과는
`../reports/bbox_repair/audit_before/summary.json`에 보관했다.
데이터를 추가하거나 좌표를 다시 변경하면 전체 검사를 재실행해야 한다.
